# Extract invoice fields with Azure Content Understanding

This notebook creates a synthetic invoice PDF, extracts business fields with `prebuilt-invoice`, and saves operation handles before polling. It uses SynapseML's direct REST integration, not the Content Understanding Python SDK.

**Prerequisites:** a Spark session named `spark`, matching SynapseML Python and JVM artifacts containing `ContentUnderstanding`, and an authorized Azure Content Understanding resource with its required model deployments configured. In Fabric, attach a lakehouse. See the [SynapseML installation guide](https://microsoft.github.io/SynapseML/docs/Get%20Started/Install%20SynapseML/) and [Content Understanding model setup](https://learn.microsoft.com/azure/ai-services/content-understanding/concepts/models-deployments).

Analysis is billable. Use only a resource and output path you are authorized to use. This notebook does not provision analyzers, deploy models, or change shared resource defaults. Keep the same document ID, configuration, and journal path when resuming.

In [ ]:
import os

from pyspark.sql import functions as F
from synapse.ml.services.contentunderstanding import ContentUnderstanding

endpoint = os.environ["CONTENT_UNDERSTANDING_ENDPOINT"]
key = os.environ["CONTENT_UNDERSTANDING_API_KEY"]
# In Fabric, retrieve the key with notebookutils.credentials.getSecret instead.
journal_path = os.environ.get(
    "CONTENT_UNDERSTANDING_OUTPUT_PATH",
    "Files/content-understanding/invoice-demo-v1",
)
journal_format = "parquet"  # Use "delta" in a Delta-enabled lakehouse.

analyzer = (
    ContentUnderstanding()
    .setEndpoint(endpoint)
    .setSubscriptionKey(key)
    .setApiVersion("2025-11-01")
    .setAnalyzerId("prebuilt-invoice")
    .setDocumentBytesCol("body")
    .setDocumentName("synthetic-invoice.pdf")
    .setMimeType("application/pdf")
    .setRetryOnThrottling(True)
    .setPollingDelay(1000)
    .setMaxPollAttempts(120)
)

Never print the key or save an authenticated SparkML stage: stage persistence includes scalar credential parameters. For Entra ID, replace `setSubscriptionKey` with `setAADToken` and provide a fresh token for `https://cognitiveservices.azure.com/.default`.

The example uses the resource's configured model defaults. If needed, set `modelDeployments` to a map from `prebuilt-analyzer-completion` and `prebuilt-analyzer-embedding` to your supported deployment names. Those per-request mappings do not change resource defaults.

## Create a deterministic document

The PDF below contains no customer data and requires no document-generation package. Its bytes and versioned ID stay the same when the notebook is rerun.

In [ ]:
def invoice_pdf():
    text = (
        "BT /F1 16 Tf 50 730 Td (SYNTHETIC INVOICE) Tj "
        "0 -35 Td (Vendor: Contoso Example Supplies) Tj "
        "0 -35 Td (Customer: Fabrikam Example Company) Tj "
        "0 -35 Td (Invoice number: CU-DEMO-001) Tj "
        "0 -35 Td (Invoice date: 2026-01-15) Tj "
        "0 -35 Td (Widgets: 2 x 10.00 USD) Tj "
        "0 -35 Td (Total amount: 20.00 USD) Tj ET"
    ).encode("ascii")
    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [4 0 R] /Count 1 >>",
        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
        b"/Resources << /Font << /F1 3 0 R >> >> /Contents 5 0 R >>",
        f"<< /Length {len(text)} >>\nstream\n".encode("ascii") + text + b"\nendstream",
    ]
    data = bytearray(b"%PDF-1.4\n")
    offsets = [0]
    for number, obj in enumerate(objects, 1):
        offsets.append(len(data))
        data.extend(f"{number} 0 obj\n".encode("ascii") + obj + b"\nendobj\n")
    xref = len(data)
    data.extend(f"xref\n0 {len(offsets)}\n0000000000 65535 f \n".encode("ascii"))
    for offset in offsets[1:]:
        data.extend(f"{offset:010d} 00000 n \n".encode("ascii"))
    data.extend(
        f"trailer\n<< /Size {len(offsets)} /Root 1 0 R >>\n"
        f"startxref\n{xref}\n%%EOF\n".encode("ascii")
    )
    return data


documents = spark.createDataFrame(
    [("synthetic-invoice-v1", invoice_pdf())], "documentId string, body binary"
)

## Submit and save the operation handle

`submit` mode returns after the service accepts the document and its handle is committed. The service processes the document asynchronously; no detached polling thread keeps running in the notebook. Repeating this cell with the same manifest and journal skips already-saved handles.

Only one writer may own this journal. There is still a crash window between service acceptance and committing the handle; this is not an exactly-once billing guarantee.

In [ ]:
analyzer.setOperationMode("submit").writeToPath(
    documents, "documentId", journal_path, format=journal_format
)
saved = analyzer.readPath(spark, journal_path, journal_format)
saved.select("documentId", "status", "httpStatus").show(truncate=False)

## Resume and commit the result

You can run this cell immediately or in a later session after recreating the same configuration and document. It polls the saved handle instead of submitting another document. Completed records are skipped. Resume promptly: service results have limited retention.

If the service is still running after `maxPollAttempts`, rerun this cell. HTTP 429 responses are retried indefinitely by default without consuming that budget. This driver-side writer stops retries on JVM thread interruption; a notebook interrupt that only stops Python waiting may not stop that call. Use `setRetryOnThrottling(False)` for bounded polling attempts. Other transport failures retain the journal's accepted handle for recovery.

In [ ]:
analyzer.setOperationMode("analyze").writeToPath(
    documents, "documentId", journal_path, format=journal_format
)
latest = analyzer.readPath(spark, journal_path, journal_format)
latest.select("documentId", "status", "httpStatus", "error").show(truncate=False)
assert (
    latest.where("status = 'Succeeded'").count() == 1
), "Inspect status/error above. Resume pending work; do not treat it as an empty result."

## Read semantic fields, confidence, and grounding

These are invoice-schema fields, not just OCR text. `rawResponse` preserves the full service JSON. The official [invoice schema](https://github.com/Azure/content-understanding-toolkit/blob/main/prebuilt-schema/2025-11-01/procurement/invoice.md) defines `VendorName`, `InvoiceId`, and the nested `TotalAmount.Amount`.

Confidence and source metadata may be absent for some fields; retain nulls rather than inventing confidence. The table below reads only the saved journal, so displaying it repeatedly does not call the service.

In [ ]:
fields_path = "$.result.contents[0].fields"


def field(path):
    return F.get_json_object("rawResponse", f"{fields_path}.{path}")


invoices = latest.where("status = 'Succeeded'").select(
    "documentId",
    field("VendorName.valueString").alias("vendor"),
    field("VendorName.confidence").cast("double").alias("vendor_confidence"),
    field("VendorName.source").alias("vendor_source"),
    field("InvoiceId.valueString").alias("invoice_id"),
    field("TotalAmount.valueObject.Amount.valueNumber").cast("double").alias("total"),
    field("TotalAmount.valueObject.Amount.confidence")
    .cast("double")
    .alias("total_confidence"),
    field("TotalAmount.valueObject.Amount.source").alias("total_source"),
    F.get_json_object("rawResponse", "$.usage").alias("usage"),
)
invoices.show(truncate=False)
assert (
    invoices.where(
        "vendor IS NOT NULL AND invoice_id IS NOT NULL AND total IS NOT NULL"
    ).count()
    == 1
)

## Concurrency, preview options, and cleanup

For parallel Spark pipelines, use `transform` with `setConcurrency(n)`: the limit applies per active partition. Persist its output once, then read that output for display; repeated actions on a lazy transform can repeat billable submissions. The durable writer used here is sequential and does not use this concurrency setting.

For `2026-06-01-preview`, `setAllowInputTruncation(False)` explicitly disables truncation. Leaving this optional parameter unset uses the analyzer's setting; nullable Boolean columns are also supported. Changing analysis options changes the request fingerprint, so use a new versioned document ID and journal for an intentional new analysis.

Keep the journal if you need recovery or the original response. Delete only your chosen `journal_path` after its data is no longer needed; this notebook never deletes it automatically. No analyzer or deployment was created.

References: [SynapseML guide](https://microsoft.github.io/SynapseML/docs/Explore%20Algorithms/AI%20Services/Content%20Understanding/), [REST analyze API](https://learn.microsoft.com/rest/api/contentunderstanding/content-analyzers/analyze?view=rest-contentunderstanding-2026-06-01-preview), [Python SDK API reference](https://learn.microsoft.com/python/api/azure-ai-contentunderstanding/azure.ai.contentunderstanding.contentunderstandingclient), and [service limits](https://learn.microsoft.com/azure/ai-services/content-understanding/service-limits).